# Performance Relation

按**试次**维度，分析AI调用行为特征与创造性绩效的关系。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from pathlib import Path

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

cluster_name_map = {
    0: '广泛试探型',
    1: '高效采纳型',
    2: '审慎批判型',
}

In [ ]:
# 输入文件路径
output_dir = Path('output')
trial_with_cluster_file = output_dir / 'trial_with_cluster.csv'

data_dir = Path('../../../data/analysis')
performance_file = data_dir / 'performance' / 'performance.csv'

In [ ]:
# 工具函数

# ---- GLMM 两两比较（基于固定效应估计，在对数链接尺度上）----
# 说明：针对前面拟合的 PoissonBayesMixedGLM（fluency 与 above_median），
# 计算不同 cluster 水平之间的两两对比：估计差值（link），标准误，z / p，
# 以及对数链接尺度的 95% 置信区间与 IRR（exp(diff)）区间。
import patsy
from itertools import combinations
from statsmodels.stats.multitest import multipletests
import numpy as np
from scipy import stats as _sps
import pandas as _pd

def pairwise_glmm_contrasts(glmm, glmm_res, df, formula='C(cluster) + dat_score + ribs_total + cse_total + ai_attitude', cluster_col='cluster', covariates=None, model_label='model'):
    exog_names = list(glmm.exog_names)
    k_fe = len(exog_names)

    # 获取固定效应及其协方差：VB 有 fe_mean / fe_sd，MAP 有 params / cov_params
    if hasattr(glmm_res, 'fe_mean'):
        fe_mean = np.asarray(glmm_res.fe_mean).reshape(-1)
        fe_sd = np.asarray(glmm_res.fe_sd).reshape(-1)
        cov_fe = np.diag(fe_sd ** 2)
        cov_fe = _pd.DataFrame(cov_fe, index=exog_names, columns=exog_names)
    else:
        all_params = np.asarray(glmm_res.params).reshape(-1)
        fe_mean = all_params[:k_fe]
        try:
            cov = np.asarray(glmm_res.cov_params())
            cov_fe = _pd.DataFrame(cov, index=glmm.exog_names, columns=glmm.exog_names)
            cov_fe = cov_fe.reindex(index=exog_names, columns=exog_names).fillna(0.0)
        except Exception:
            cov_fe = _pd.DataFrame(np.nan, index=exog_names, columns=exog_names)

    # cluster 水平
    if isinstance(df[cluster_col].dtype, pd.CategoricalDtype):
        clusters = df[cluster_col].cat.categories
    else:
        clusters = sorted(df[cluster_col].dropna().unique())

    pred_df = _pd.DataFrame({cluster_col: clusters})
    # 填充协变量（默认使用样本均值），covariates 可以传入 dict 覆盖
    if covariates is None:
        covariates = {}
    for cov, val in covariates.items():
        if val is None and cov in df.columns:
            pred_df[cov] = df[cov].mean()
        else:
            pred_df[cov] = val

    # 若连续协变量未传且存在于 df 中，则设为均值
    for cov in ['dat_score', 'ribs_total', 'cse_total', 'ai_attitude']:
        if cov not in pred_df.columns and cov in df.columns:
            pred_df[cov] = df[cov].mean()

    design = patsy.dmatrix(formula, pred_df, return_type='dataframe')
    # 保证 design 列与 exog_names 对齐
    for col in exog_names:
        if col not in design.columns:
            design[col] = 0.0
    design = design[exog_names]

    results = []
    for i, j in combinations(range(len(clusters)), 2):
        xa = design.iloc[i].values
        xb = design.iloc[j].values
        contrast = xa - xb
        diff = float(contrast.dot(fe_mean))
        # 计算方差（若协方差包含 NaN 则返回 NaN）
        try:
            var = float(contrast.dot(cov_fe.values).dot(contrast))
        except Exception:
            var = np.nan
        se = np.sqrt(var) if not np.isnan(var) else np.nan
        zstat = diff / se if (se and not np.isnan(se)) else np.nan
        pval = 2 * _sps.norm.sf(abs(zstat)) if not np.isnan(zstat) else np.nan
        # 95% CI 在 link（对数）尺度上，再转换到 IRR（exp）尺度
        if not np.isnan(se):
            ci_lo = diff - 1.96 * se
            ci_hi = diff + 1.96 * se
            irr = np.exp(diff)
            irr_lo = np.exp(ci_lo)
            irr_hi = np.exp(ci_hi)
        else:
            ci_lo = ci_hi = irr = irr_lo = irr_hi = np.nan
        results.append({
            'group1': clusters[i],
            'group2': clusters[j],
            'est_link': diff,
            'se': se,
            'z': zstat,
            'p': pval,
            'ci_lo_link': ci_lo,
            'ci_hi_link': ci_hi,
            'irr': irr,
            'irr_lo': irr_lo,
            'irr_hi': irr_hi
        })

    res_df = _pd.DataFrame(results).sort_values('p')
    if len(res_df):
        res_df['p_bonf'] = multipletests(res_df['p'].values, method='bonferroni')[1]
        res_df['p_fdr'] = multipletests(res_df['p'].values, method='fdr_bh')[1]
    print(f'--- {model_label} 两两比较（按 p 排序）---')
    display(res_df.round(4))
    return res_df


In [ ]:
# 加载数据
df_cluster = pd.read_csv(trial_with_cluster_file)  # 包含 participant_id, trial_index, cluster 与行为特征
df_perf = pd.read_csv(performance_file)             # 包含 participant_id, item_name, trial_index, fluency, originality

# 以试次为单位合并聚类标签
df_merged = pd.merge(
    df_perf,
    df_cluster,
    on=['participant_id', 'trial_index'],
    how='left'
 )

print(df_merged['cluster'].value_counts(dropna=False).sort_index())
df_merged.describe()

## 1. 调用行为特征与发散思维绩效的相关性

In [ ]:
# 筛选有AI调用的试次
df_called = df_merged[df_merged['cluster'] >= 0].copy()

# 计算相关系数矩阵
corr_vars = [
    # 行为特征
    'trial_calls',
    'first_ai_time',
    'pre_first_call_ideas',
    'pre_think_time',
    'perspective_taking',
    'affected_by_ai',
    # 人格特质
    # 'bfi_extroversion',
    # 'bfi_agreeableness',
    # 'bfi_conscientiousness',
    # 'bfi_neuroticism',
    # 'bfi_openness',
    'cse_total',
    'ribs_total',
    # 'ai_attitude',
    'dat_score',
    # 创造性表现
    'originality',
    'fluency',
    'above_median'
]
corr_matrix = df_called[corr_vars].corr(method='pearson')

# 中文标签映射
corr_labels_cn = {
    'trial_calls': 'AI调用次数',
    'first_ai_time': '首次调用时间',
    'pre_first_call_ideas': '首次调用前想法数',
    'pre_think_time': '平均调用前思考时间',
    'perspective_taking': '观点采择率',
    'affected_by_ai': '受AI影响率',
    'bfi_extroversion': '外向性',
    'bfi_agreeableness': '宜人性',
    'bfi_conscientiousness': '尽责性',
    'bfi_neuroticism': '神经质',
    'bfi_openness': '开放性',
    'cse_total': '创造性自我效能',
    'ribs_total': 'RIBS总分',
    'ai_attitude': 'AI态度',
    'dat_score': 'DAT分数',
    'originality': '原创性',
    'fluency': '流畅性',
    'above_median': '高质量回答数',
    'above_median_ratio': '高质量回答比率',
}
corr_vars_cn = [corr_labels_cn[v] for v in corr_vars]
corr_matrix_cn = corr_matrix.rename(index=corr_labels_cn, columns=corr_labels_cn)

In [ ]:
from scipy.stats import pearsonr

# 1. 计算 P 值并构造标注文本矩阵
annot_matrix = pd.DataFrame(index=corr_vars_cn, columns=corr_vars_cn)

for i, col1 in enumerate(corr_vars):
    for j, col2 in enumerate(corr_vars):
        label1, label2 = corr_vars_cn[i], corr_vars_cn[j]
        # 对角线或自身
        if col1 == col2:
            annot_matrix.loc[label1, label2] = f"{corr_matrix_cn.loc[label1, label2]:.2f}"
            continue
            
        # 提取两列数据并去除空值（确保对齐）
        df_pair = df_called[[col1, col2]].dropna()
        
        if len(df_pair) > 1:
            r, p = pearsonr(df_pair[col1], df_pair[col2])
            
            # 根据 P 值添加星号
            if p < 0.001:
                stars = '***'
            elif p < 0.01:
                stars = '**'
            elif p < 0.05:
                stars = '*'
            else:
                stars = ''
            
            annot_matrix.loc[label1, label2] = f"{r:.2f}{stars}"
        else:
            annot_matrix.loc[label1, label2] = "NaN"

# 2. 绘制热力图
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix_cn, annot=annot_matrix.values, cmap='coolwarm', fmt='', vmin=-1, vmax=1)
plt.title('调用行为特征与发散思维绩效的相关性')
plt.tight_layout()
plt.show()

In [ ]:
# 显著性检验
FIRST_THRESHOLD = 0.05
SECOND_THRESHOLD = 0.01
THIRD_THRESHOLD = 0.001

from scipy.stats import pearsonr
p_values = {}
for col in corr_vars:
    for col2 in corr_vars:
        if col == col2:
            continue
        pair_df = df_called[[col, col2]].dropna()
        if len(pair_df) > 1:
            r, p = pearsonr(pair_df[col], pair_df[col2])
            p_values[f'{col}-{col2}'] = p
        else:
            p_values[f'{col}-{col2}'] = float('nan')

# 构建 p 值矩阵（DataFrame），使用中文标签
import numpy as np
p_matrix = pd.DataFrame(index=corr_vars_cn, columns=corr_vars_cn, dtype=float)
for i, col in enumerate(corr_vars):
    for j, col2 in enumerate(corr_vars):
        label1, label2 = corr_vars_cn[i], corr_vars_cn[j]
        if col == col2:
            p_matrix.loc[label1, label2] = np.nan
        else:
            key = f'{col}-{col2}'
            p_matrix.loc[label1, label2] = p_values.get(key, np.nan)

# 生成带星号的注释矩阵
def annot_formatter(p):
    if pd.isna(p):
        return ''
    elif p < THIRD_THRESHOLD:
        return f'{p:.3f}***'
    elif p < SECOND_THRESHOLD:
        return f'{p:.3f}**'
    elif p < FIRST_THRESHOLD:
        return f'{p:.3f}*'
    else:
        return f'{p:.3f}'

annot_matrix = p_matrix.apply(lambda col: col.map(annot_formatter))

# 绘制热力图
plt.figure(figsize=(12, 10))
sns.heatmap(p_matrix.astype(float),
            annot=annot_matrix,
            fmt='',
            cmap='coolwarm_r',
            cbar_kws={'label': 'p-value'})
plt.title('相关显著性 p 值')
plt.tight_layout()
plt.show()

In [ ]:
# FDR 校正（Benjamini-Hochberg 方法）
# 对 Pearson 相关矩阵中的 153 对（18×17/2）非重复相关进行多重比较校正

from statsmodels.stats.multitest import multipletests

# 1. 收集所有唯一的 pairwise p 值（下三角，不含对角线）
p_flat = []
p_labels = []
r_flat = []
for i, col1 in enumerate(corr_vars):
    for j, col2 in enumerate(corr_vars):
        if i >= j:
            continue
        pair_df = df_called[[col1, col2]].dropna()
        if len(pair_df) > 1:
            r, p = pearsonr(pair_df[col1], pair_df[col2])
            p_flat.append(p)
            r_flat.append(r)
            p_labels.append(f'{corr_labels_cn[col1]}  vs  {corr_labels_cn[col2]}')

# 2. 应用 Benjamini-Hochberg FDR 校正
reject_fdr, p_fdr, _, _ = multipletests(p_flat, method='fdr_by')

# 3. 构建 FDR 校正后的 p 值矩阵（对称填充），使用中文标签
p_fdr_full = pd.DataFrame(np.nan, index=corr_vars_cn, columns=corr_vars_cn)
reject_fdr_full = pd.DataFrame(False, index=corr_vars_cn, columns=corr_vars_cn)
idx = 0
for i, col1 in enumerate(corr_vars):
    for j, col2 in enumerate(corr_vars):
        if i >= j:
            continue
        label1, label2 = corr_vars_cn[i], corr_vars_cn[j]
        p_fdr_full.loc[label1, label2] = p_fdr[idx]
        p_fdr_full.loc[label2, label1] = p_fdr[idx]
        reject_fdr_full.loc[label1, label2] = reject_fdr[idx]
        reject_fdr_full.loc[label2, label1] = reject_fdr[idx]
        idx += 1

# 4. 输出校正前显著但 FDR 校正后不显著的相关
print("校正前显著但 FDR 校正后不显著的相关（p_raw < 0.05 且 p_fdr >= 0.05）：")
print("-" * 70)
n_sig_lost = 0
n_sig_orig = 0
n_sig_remain = 0
for k, (label, r_val, p_raw, p_corr, rej) in enumerate(zip(p_labels, r_flat, p_flat, p_fdr, reject_fdr)):
    if p_raw < 0.05:
        n_sig_orig += 1
        if not rej:
            n_sig_lost += 1
            stars = '***' if p_raw < 0.001 else ('**' if p_raw < 0.01 else '*')
            print(f"  {label}: r = {r_val:+.3f}, p_raw = {p_raw:.4f} {stars}, p_fdr = {p_corr:.4f} (n.s.)")
        else:
            n_sig_remain += 1

print(f"\n{len(p_flat)} 对比较中，原始有 {n_sig_orig} 对显著。经过 FDR 校正后：")
print(f" - {n_sig_remain} 对保持显著")
print(f" - {n_sig_lost} 对失去显著性（如上列出）")

# 5. 绘制 FDR 校正后的热力图：保留全部色彩，仅对 FDR 显著的单元格标注星号
annot_fdr = pd.DataFrame('', index=corr_vars_cn, columns=corr_vars_cn)
for i, col1 in enumerate(corr_vars):
    for j, col2 in enumerate(corr_vars):
        label1, label2 = corr_vars_cn[i], corr_vars_cn[j]
        if i == j:
            annot_fdr.loc[label1, label2] = f"{corr_matrix_cn.loc[label1, label2]:.2f}"
        elif reject_fdr_full.loc[label1, label2]:
            r_val = corr_matrix_cn.loc[label1, label2]
            p_val = p_fdr_full.loc[label1, label2]
            stars = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else '*')
            annot_fdr.loc[label1, label2] = f"{r_val:.2f}{stars}"

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix_cn, annot=annot_fdr.values, cmap='coolwarm', fmt='', vmin=-1, vmax=1)
plt.title('行为指标、个人特质与发散思维绩效的相关性（FDR 校正后）')
plt.tight_layout()
plt.show()

In [ ]:
# 输出FDR校正后仍然显著的指标

# FDR 校正后仍然显著的相关（p_fdr < 0.05）
sig_mask = reject_fdr
n_sig = sig_mask.sum()
print(f'FDR 校正后仍然显著的相关（共 {n_sig} 对）：')
print('-' * 70)

# 按 r 绝对值降序排列
sig_results = []
for k, (label, r_val, p_raw, p_corr, rej) in enumerate(zip(p_labels, r_flat, p_flat, p_fdr, reject_fdr)):
    if rej:
        sig_results.append({
            'pair': label,
            'r': r_val,
            'p_raw': p_raw,
            'p_fdr': p_corr
        })

sig_df = pd.DataFrame(sig_results)
sig_df['abs_r'] = sig_df['r'].abs()
sig_df = sig_df.sort_values('abs_r', ascending=False).drop(columns=['abs_r'])

for _, row in sig_df.iterrows():
    r_val = row['r']
    pair = row['pair']
    p_raw = row['p_raw']
    p_fdr_val = row['p_fdr']
    if p_fdr_val < 0.001:
        stars = '***'
    elif p_fdr_val < 0.01:
        stars = '**'
    else:
        stars = '*'
    print(f'  {pair}: r = {r_val:+.3f}, p_raw = {p_raw:.4f}, p_fdr = {p_fdr_val:.4f} {stars}')

print()
print(f'FDR 校正前后对比：')
print(f'  校正前显著（p_raw < .05）：{n_sig_orig} 对')
print(f'  校正后显著（p_fdr < .05）：{n_sig} 对')
print(f'  失去显著性：{n_sig_lost} 对')

print()
print('--- 按行为指标分组的 FDR 显著相关 ---')
behav_vars_cn = ['AI调用次数', '首次调用时间', '首次调用前想法数', '平均调用前思考时间', '观点采择率', '受AI影响率']
for bv in behav_vars_cn:
    subset = sig_df[sig_df['pair'].str.startswith(bv)]
    if len(subset) > 0:
        print()
        print(f'  [{bv}]:')
        for _, row in subset.iterrows():
            r_val = row['r']
            pair = row['pair']
            p_fdr_val = row['p_fdr']
            if p_fdr_val < 0.001:
                stars = '***'
            elif p_fdr_val < 0.01:
                stars = '**'
            else:
                stars = '*'
            print(f'    {pair}: r = {r_val:+.3f}, p_fdr = {p_fdr_val:.4f} {stars}')


In [ ]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
import numpy as np
import pandas as pd

# 通过机器学习（如SVM）将多项行为指标、个人特质等特征作为输入，预测最终的绩效指标值（如originality）
def predict_performance_with_ml(df, features, target='originality'):
    # Drop rows where target is missing
    data = df.dropna(subset=[target]).copy()
    
    X = data[features]
    y = data[target]
    
    # Define models to compare
    models = {
        'Ridge': Ridge(),
        'SVR': SVR(kernel='rbf', C=1.0, epsilon=0.1),
        'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42)
    }
    
    print(f"=== Predicting '{target}' using {len(features)} features ===")
    print(f"Sample size: {len(data)}")
    
    results = {}
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    for name, model in models.items():
        # Pipeline: Impute missing -> Scale -> Predict
        pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('regressor', model)
        ])
        
        scores = cross_validate(pipeline, X, y, cv=kf, 
                                scoring=['neg_mean_squared_error', 'r2', 'neg_mean_absolute_error'],
                                n_jobs=-1)
        
        mse = -np.mean(scores['test_neg_mean_squared_error'])
        r2 = np.mean(scores['test_r2'])
        mae = -np.mean(scores['test_neg_mean_absolute_error'])
        
        results[name] = {'MSE': mse, 'RMSE': np.sqrt(mse), 'MAE': mae, 'R2': r2}
        print(f"\nModel: {name}")
        print(f"  MSE : {mse:.4f}")
        print(f"  RMSE: {np.sqrt(mse):.4f}")
        print(f"  MAE : {mae:.4f}")
        print(f"  R2  : {r2:.4f}")
        
    return pd.DataFrame(results).T

In [ ]:
features = ['trial_calls', 'first_ai_time', 'pre_first_call_ideas', 'pre_think_time', 'perspective_taking', 'affected_by_ai', 'dat_score', 'ai_attitude']
res = predict_performance_with_ml(df_called, features, target='originality')